<a href="https://colab.research.google.com/github/Ali-Shahrez/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding #4 — "The Freshness Multiplier"

The claim: 365+ day content that was refreshed within 30 days shows a 3.2x health boost (10.7 → 34.5) and 57x more impressions (71 → 4039). The paper calls refresh timing "one of the strongest measured levers available."

Where does the label come from? "Refreshed" is operationalized purely as time elapsed since some update field changed — the paper is silent on what write-event moves that field or who/what triggers it.

Was the refresh group chosen, or random? Almost certainly chosen, and chosen in a way that stacks the deck. An editor (or FlyRank's own optimization-flag system, which the report describes) is going to prioritize pages that already show signs of life — some baseline traffic, a relevant topic, existing authority — nobody spends an afternoon rewriting a page getting zero impressions on a dead topic. The report's own words support this: Finding #1 describes declining pages as "still carry meaningful impressions, but... older, thinner, and less likely to retain momentum" — exactly the cohort an editor would flag for refresh, not a random sample of old content. Myth #3 says flags "can only trigger on pages that already have enough impressions or behavior data to diagnose a concrete issue." So the refreshed and non-refreshed groups are systematically different populations before anyone touched anything.

Does the validation design carry the claim? No. "Refreshing produces a 3.2x boost" is a causal claim about an intervention — what a given page's score would have been had it not been refreshed. A snapshot comparison of refreshed-vs-not can't support that; it's the wrong design structurally. What the sentence would need: random or as-good-as-random assignment to refresh vs. no-refresh, a pre/post comparison on the same pages rather than a cross-sectional split, and a matched or synthetic control for what the refreshed pages' trajectory would have looked like left alone — ideally with staggered refresh timing so "not yet refreshed" pages can stand in as controls.

Honest, weaker claim: In this portfolio, pages selected for refresh — by editors or by FlyRank's flagging logic, which tend to target pages with pre-existing visibility and recoverable problems — currently show much higher health scores than old pages that were never selected. This is consistent with refresh effort being well-targeted and not wasted, but the design can't separate "refresh caused the lift" from "pages likely to perform well were the ones chosen to refresh."

Finding #7 — "The Winning Combinations"

The claim: transactional/LOW-competition pages lead on health score (30 vs. 21-25 for other intent×competition cells); among page-1 pages, the 3500+ word tier shows higher health (44.8) than the shortest tier (40.1).

Where does the label come from? Same instrument as Finding #4: health score, the composite of impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts). The ML appendix's own Random Forest importance ranking shows Average Position (43%) and Impressions (32%) as the top predictors of health score — roughly 90% of what "explains" the score is two of its own four ingredients. The report even names this itself: "health score is partly constructed from inputs such as position and impressions... high importance is therefore expected and does not imply external causation." So when Finding #7 says transactional/LOW pages "win," that could be re-reporting that those pages rank better and get more impressions — largely circular, since ranking and impressions are most of the score. The correlation matrix undercuts the finding's own framing further: competition's correlation with health score is listed at ~0.0, and the health gap it leans on (23.8 vs. 22.9, "only 0.9 points" by the report's own admission) is small enough to be noise in a mostly-circular metric. The more defensible number in that section is the growth ratio (2.1:1 vs. 1.3:1) — an independently-measured outcome, not a health-score artifact. There's also a selection layer here too: intent and competition aren't randomly assigned to pages — an editor chose to target "transactional, low-competition" for some pages and "informational, high-competition" for others, the same selection logic as Finding #4, just operating at topic-selection time instead of refresh time.

Does the validation design carry the claim, for the page-1 word-count table specifically? The report's own caption gives away the problem: "This second chart only looks at pages already on page 1." Position is 30 of 100 health-score points and the single strongest predictor of it (43% RF importance, r = -0.59, the strongest correlation in the whole matrix) — so "pages already on page 1" is functionally "pages that already scored well on the position component of health score." Conditioning on that before asking whether word count still predicts health is closer to a collider/Berkson's-paradox setup than a clean comparison: restricting to an already-successful subgroup can produce, shrink, or reverse a relationship that looks different in the full population. There's no way to tell from this table whether longer content helps pages reach page 1, helps pages already on page 1 climb higher (which would mechanically move health score through the position component, not through anything word count does independently), or whether topics that naturally need 3500+ words are just different topics that would have scored higher regardless of length — and Finding #1 already established that word count and topic maturity aren't independent.

Honest, weaker claim: Among pages that have already reached page 1 — which is itself likely a function of the same depth/quality that produced the page-1 result — the longest tier carries a modestly higher health score, but that gap isn't separable from the position and topic-maturity effects built into both the filter and the metric. It doesn't support "add words to boost health," even for page-one pages, without knowing whether length caused the good position or just correlates with the kind of comprehensive page that tends to earn one.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
%cd /content
!rm -rf flyrank-ml-internship
!git clone -q https://github.com/Ali-Shahrez/flyrank-ml-internship.git
%cd flyrank-ml-internship

/content
/content/flyrank-ml-internship


In [3]:
# --- Load data + reproduce the canonical GROUPED split (Week 5's method) ---
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
MIN_IMPRESSIONS = 500

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows")

# canonical grouped (client-holdout) split — same method as w05
client_series = df['client_id'].fillna('unknown').astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = client_series.isin(test_clients).to_numpy()

grouped_train_df = df[~test_mask].copy()
grouped_test_df  = df[test_mask].copy()

grouped_train_elig = grouped_train_df[grouped_train_df['impressions_90d'] >= MIN_IMPRESSIONS].copy()
grouped_test_elig  = grouped_test_df[grouped_test_df['impressions_90d'] >= MIN_IMPRESSIONS].copy()
for f in (grouped_train_elig, grouped_test_elig):
    f['is_declining_label'] = (f['trend_direction'] == 'down').astype(int)

print(f"[GROUPED] train {len(grouped_train_elig):,} / test {len(grouped_test_elig):,} "
      f"/ test base rate {grouped_test_elig['is_declining_label'].mean():.3f}")

# --- Build the NAIVE RANDOM split (row-level, ignores client_id) ---
df_elig = df[df['impressions_90d'] >= MIN_IMPRESSIONS].copy()
df_elig['is_declining_label'] = (df_elig['trend_direction'] == 'down').astype(int)

# match test size to the grouped split's test-eligible count, so N is comparable
test_frac = len(grouped_test_elig) / (len(grouped_train_elig) + len(grouped_test_elig))
print(f"Target test fraction (matched to grouped split): {test_frac:.3f}")

naive_train_elig, naive_test_elig = train_test_split(
    df_elig, test_size=test_frac, random_state=RANDOM_STATE, stratify=df_elig['is_declining_label']
)

print(f"[NAIVE RANDOM] train {len(naive_train_elig):,} / test {len(naive_test_elig):,} "
      f"/ test base rate {naive_test_elig['is_declining_label'].mean():.3f}")

# how many clients appear in BOTH naive train and naive test — this is the leakage mechanism
naive_train_clients = set(naive_train_elig['client_id'])
naive_test_clients = set(naive_test_elig['client_id'])
overlap = naive_train_clients & naive_test_clients
print(f"[NAIVE RANDOM] clients in both train and test: {len(overlap)} of {len(naive_test_clients)} test clients")

Loaded 30,000 rows
[GROUPED] train 16,108 / test 618 / test base rate 0.519
Target test fraction (matched to grouped split): 0.037
[NAIVE RANDOM] train 16,108 / test 618 / test base rate 0.595
[NAIVE RANDOM] clients in both train and test: 22 of 22 test clients


In [4]:
import sys, os
sys.path.append(os.path.abspath('scripts'))
from ml_utils import precision_at_k, MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

MISSINGNESS_TRACKED_COLS = ['word_count', 'char_count', 'search_volume', 'competition', 'cpc']

def add_derived_and_flags(frame):
    frame = frame.copy()
    for col in MISSINGNESS_TRACKED_COLS:
        frame[f'has_{col}'] = frame[col].notna().astype(int)
    frame['log_impressions_90d'] = np.log1p(frame['impressions_90d'])
    frame['log_clicks_90d']      = np.log1p(frame['clicks_90d'])
    frame['log_sessions_90d']    = np.log1p(frame['sessions_90d'])
    frame['log_ai_sessions_90d'] = np.log1p(frame['ai_sessions_90d'])
    return frame

def build_features(frame, numeric_cols, categorical_cols, flag_cols):
    numeric = frame[numeric_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    flags = frame[flag_cols]
    categorical = frame[categorical_cols].fillna('unknown').astype(str)
    dummies = pd.get_dummies(categorical, prefix=categorical_cols, dtype=float)
    return pd.concat([numeric.reset_index(drop=True), flags.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)

def run_split(train_df, test_df, label):
    train_df = add_derived_and_flags(train_df)
    test_df  = add_derived_and_flags(test_df)
    numeric_cols = [c for c in MODEL_NUMERIC_FEATURES if c in train_df.columns]
    categorical_cols = [c for c in MODEL_CATEGORICAL_FEATURES if c in train_df.columns]
    flag_cols = [f'has_{c}' for c in MISSINGNESS_TRACKED_COLS]

    X_train = build_features(train_df, numeric_cols, categorical_cols, flag_cols)
    X_test  = build_features(test_df,  numeric_cols, categorical_cols, flag_cols)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    y_train = train_df['is_declining_label']
    y_test  = test_df['is_declining_label']

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    logreg.fit(X_train_scaled, y_train)
    logreg_scores = logreg.predict_proba(X_test_scaled)[:, 1]

    rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                                 min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
    rf.fit(X_train, y_train)
    rf_scores = rf.predict_proba(X_test)[:, 1]

    return {
        'split': label, 'n_train': len(X_train), 'n_test': len(X_test),
        'test_base_rate': round(y_test.mean(), 3),
        'logreg_p20': precision_at_k(y_test, logreg_scores, 20),
        'logreg_p50': precision_at_k(y_test, logreg_scores, 50),
        'rf_p20': precision_at_k(y_test, rf_scores, 20),
        'rf_p50': precision_at_k(y_test, rf_scores, 50),
    }, rf, X_train, y_train, X_test, y_test, rf_scores

before_result, _, _, _, _, _, _ = run_split(naive_train_elig, naive_test_elig, 'BEFORE (naive random)')
after_result, rf_honest, X_train_h, y_train_h, X_test_h, y_test_h, rf_scores_h = run_split(grouped_train_elig, grouped_test_elig, 'AFTER (grouped, canonical)')

before_after = pd.DataFrame([before_result, after_result])
print(before_after.to_string(index=False))

                     split  n_train  n_test  test_base_rate  logreg_p20  logreg_p50  rf_p20  rf_p50
     BEFORE (naive random)    16108     618           0.595        0.90        0.84    0.95    0.88
AFTER (grouped, canonical)    16108     618           0.519        0.55        0.48    0.85    0.86


The gap, and why it's there.

| Method | Before (naive random) | After (grouped, honest) | Gap |
|---|---:|---:|---:|
| Logistic Regression P@20 | 0.90 | 0.55 | **-0.35** |
| Logistic Regression P@50 | 0.84 | 0.48 | **-0.36** |
| Random Forest P@20 | 0.95 | 0.85 | -0.10 |
| Random Forest P@50 | 0.88 | 0.86 | -0.02 |

I predicted Random Forest would show the bigger collapse going from naive to honest — reasoning that tree splits could carve out client-specific regions more easily than a single linear boundary. The result was the opposite: Logistic Regression's gap is 3-18x larger than Random Forest's.

The likely mechanism is scale, not thresholds. Within a single client, features like impressions_90d or word_count probably sit in a fairly narrow band — a small client's pages might cluster around a few hundred impressions, a large client's around the tens of thousands. That means client identity is smuggled into the raw magnitude of otherwise-continuous features: impressions_90d = 47,000 doesn't just carry decline information, it also silently signals "this is a big client." Logistic Regression fits one global coefficient per feature everywhere, so if scale correlates with outcome for reasons unrelated to genuine decline signal, LogReg can ride that shortcut cleanly. With train/test client overlap in the naive split, that's functionally the same leak as memorizing "which client is this row from," just mediated through feature scale instead of a direct client-ID lookup. Random Forest's bagging — each tree built on a bootstrap sample with a random feature subset per split — dilutes reliance on any single global scale cue, since no one shortcut can dominate every tree in the ensemble the way it can dominate a single linear model.

A caveat that matters more than the headline result. RF's smaller gap should not be read as "Random Forest generalizes better" or "resists client-identity leakage in general." Week 5's Section 4 already established that the honest grouped test set is 598 of 618 rows (~97%) from a single client (client_f74efabef1). That means these "honest" P@20/P@50 numbers aren't measuring generalization across a diverse set of unseen clients — they're measuring performance on essentially one held-out client, repeated across nearly every row. RF's 0.85/0.86 could reflect genuine robustness, or it could just be that tree splits happen to capture that particular client's structure well for reasons that wouldn't hold on a different held-out client. The honest, narrower claim: on this test set, Random Forest showed a smaller before/after gap than Logistic Regression — not that Random Forest resists client-identity leakage as a general property. A grouped test set with many distinct clients, each contributing a comparable number of rows, would be needed before trusting the model-comparison part of this result as a portfolio-wide finding, rather than the honest-vs-naive-split gap itself, which is real regardless of test-set composition.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [5]:
# Check 1 — banned columns: label-source and ID columns must not be in the feature matrix
banned = ['trend_direction', 'trend_pct', 'content_id', 'client_id']
present = [c for c in banned if c in X_train_h.columns]
print('Banned columns present in the honest feature matrix:', present if present else 'none')

Banned columns present in the honest feature matrix: none


In [6]:
# Check 2 — deliberately add a leaky feature back in, confirm the harness catches it
X_train_leaky = X_train_h.copy()
X_test_leaky  = X_test_h.copy()
X_train_leaky['trend_pct__LEAKY'] = grouped_train_elig['trend_pct'].to_numpy()
X_test_leaky['trend_pct__LEAKY']  = grouped_test_elig['trend_pct'].to_numpy()

rf_leaky = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                                   min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
rf_leaky.fit(X_train_leaky, y_train_h)
leaky_scores = rf_leaky.predict_proba(X_test_leaky)[:, 1]

leaky_p20 = precision_at_k(y_test_h, leaky_scores, 20)
leaky_p50 = precision_at_k(y_test_h, leaky_scores, 50)
honest_p20 = precision_at_k(y_test_h, rf_scores_h, 20)
honest_p50 = precision_at_k(y_test_h, rf_scores_h, 50)

print(f"Honest RF (no trend_pct):  P@20={honest_p20:.3f}  P@50={honest_p50:.3f}")
print(f"LEAKY RF (+ trend_pct):    P@20={leaky_p20:.3f}  P@50={leaky_p50:.3f}")

leaky_importances = pd.Series(rf_leaky.feature_importances_, index=X_train_leaky.columns).sort_values(ascending=False)
print("\nTop 5 feature importances WITH the leaky column added:")
print(leaky_importances.head(5))

Honest RF (no trend_pct):  P@20=0.850  P@50=0.860
LEAKY RF (+ trend_pct):    P@20=1.000  P@50=1.000

Top 5 feature importances WITH the leaky column added:
trend_pct__LEAKY    0.835913
content_age_days    0.019749
scroll_rate         0.012826
avg_position        0.012442
log_clicks_90d      0.011973
dtype: float64


In [7]:
honest_importances = pd.Series(rf_honest.feature_importances_, index=X_train_h.columns).sort_values(ascending=False)
print("Top 10 feature importances, HONEST Week 5 model:")
print(honest_importances.head(10))

Top 10 feature importances, HONEST Week 5 model:
content_age_days      0.107091
avg_position          0.083071
log_clicks_90d        0.077021
scroll_rate           0.076976
ctr                   0.064977
days_with_sessions    0.061195
char_count            0.045326
word_count            0.044474
age_tier_365+         0.042532
log_sessions_90d      0.041578
dtype: float64


Explicit comparison:

Leaky run: top feature = 0.836, gap to #2 (0.020) = 42x
Honest run: top feature (content_age_days) = 0.107, gap to #2 (avg_position, 0.083) = 1.3x

That's the number that matters most. Not just "no single feature dominates" in the abstract — the actual ratio is two orders of magnitude smaller than the calibrated leak signature. A leaky feature in this pipeline eats ~84% of the pie and leaves a ~40x cliff behind it; here the top feature holds ~11% and the next nine features decay gradually, not cliff-like:

0.107 → 0.083 → 0.077 → 0.077 → 0.065 → 0.061 → 0.045 → 0.044 → 0.043 → 0.042

That's a smooth, roughly-linear taper — each step down is on the order of 5–25%, never a sudden order-of-magnitude drop. Summed, the top 10 account for only ~64% of total importance, meaning over a third is spread across the remaining features. That shape — many features contributing comparable, modest amounts — is what you'd expect from a model that's actually integrating several weak-to-moderate signals (age, position, engagement, content depth) rather than one feature secretly encoding the label.

So does it pass the calibration check? Yes, on the specific thing the check was built to catch: no feature shows the 40x-dominance, nothing-else-matters signature that trend_pct produced. That's real evidence, not just an absence of suspicion — you have the leaky run as a contrast case and this profile doesn't resemble it.

What it still doesn't rule out, per the caveat from last turn: distributed leakage across correlated features. Two candidates worth a second look rather than taking on faith:

content_age_days at the top — this is plausible on its face (Finding #2 and the growth-logistic-regression coefficients both independently support age as a genuine driver), but it's worth confirming it isn't itself a downstream artifact of how trend_pct/the label was constructed (e.g., if the label window or trend calculation used age as an input anywhere upstream).
days_with_sessions — conceptually close to "days visible," which correlated with content_age_days at r=0.496 in the ML appendix's own correlation matrix. If several of these top-10 features are all partially encoding the same underlying "how long has this page existed and been active" signal rather than independent evidence, the importance table could look smooth and plausible while still being narrower — less genuinely diversified — than it appears.

Neither of those is a red flag on the order of the trend_pct case. But the honest model passing the "no single dominant feature" test is a lower bar than "definitely leakage-free" — it's evidence against the specific failure mode this test was designed to catch, not a certificate against every failure mode.

In [8]:
results = grouped_test_elig.reset_index(drop=True).copy()
results['rf_score'] = rf_scores_h
results_ranked = results.sort_values('rf_score', ascending=False).reset_index(drop=True)

cols = ['client_id', 'rf_score', 'is_declining_label', 'impressions_90d', 'avg_position',
        'ctr', 'days_since_last_update', 'content_age_days', 'trend_pct']

print('--- False positives in the top 20 (model ranked high, page was NOT declining) ---')
top20 = results_ranked.head(20)
false_positives = top20[top20['is_declining_label'] == 0]
print(f"{len(false_positives)} of the top 20\n")
print(false_positives[cols].to_string(index=False))

print('\n--- False negatives: actually declining, ranked outside the top 50 ---')
outside_top50 = results_ranked.iloc[50:]
false_negatives = outside_top50[outside_top50['is_declining_label'] == 1].sort_values('rf_score', ascending=False).head(5)
print(false_negatives[cols].to_string(index=False))

--- False positives in the top 20 (model ranked high, page was NOT declining) ---
3 of the top 20

        client_id  rf_score  is_declining_label  impressions_90d  avg_position  ctr  days_since_last_update  content_age_days  trend_pct
client_f74efabef1  0.751619                   0             3026          35.9 0.00                      20               134       64.5
client_f74efabef1  0.744184                   0              554          23.3 0.00                      20               125       40.7
client_f74efabef1  0.732427                   0             1076          25.6 0.09                      20               125       50.4

--- False negatives: actually declining, ranked outside the top 50 ---
        client_id  rf_score  is_declining_label  impressions_90d  avg_position  ctr  days_since_last_update  content_age_days  trend_pct
client_f74efabef1  0.707363                   1             1703          19.4 0.00                       8               148      -51.0
client_

Reading the false positives. All three top-20 misses share the same profile: avg_position in the mid-20s to mid-30s (weak, not page-one or reliably page-two), ctr near zero (0.00-0.09), and days_since_last_update at 20 (stale within this window). That's a reasonable inference given what the model can see — those are exactly the features that correlate with decline elsewhere in the data. The problem isn't the model's logic, it's that trend_pct for these three rows is actually strongly positive (+40.7 to +64.5) — these pages are growing, not declining, and the label (0) is correct. The model has no way to know that, because the one feature that would tell it — recent trajectory — is the same feature the label is thresholded from, so it's correctly excluded from the features by the leakage rule tested above.

Reading the false negatives. These are pages with real, steep declines (trend_pct from -42% to -95%) that the model scored just below the top-50 cutoff. Their scores (0.70-0.71) sit almost on top of the false positives' scores (0.73-0.75) — not a comfortable margin, a narrow overlapping band. That closeness is the actual finding: this isn't the model randomly misfiring on a few rows, it's the model confidently and consistently treating "weak position + low CTR + stale update" as similarly risky regardless of which direction the page is actually moving. Two pages can share that exact profile while one climbs and one falls, and the honest feature set is structurally blind to the difference.

This is a real ceiling, not a fixable bug. The feature that would resolve the ambiguity — recent trend direction — is exactly the column the leakage test in this section confirmed cannot be a feature. So the honest model can rank "generally at-risk-looking" pages reasonably well, but it has no mechanism to separate a currently-recovering weak page from a currently-declining one. That's a structural limitation of this feature set, not something more data or different hyperparameters would fix.

One more caveat, consistent with Section 2: every row in both tables belongs to client_f74efabef1. This error analysis, like the rest of the honest split's results, describes the model's behavior on one client's content specifically — not a general error profile across the portfolio.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original (Week 4, signal check verdicts): "Verdict — CTR vs. position peers: CONFIRMED."

That word choice borrows the paper's own CONFIRMED/REVERSED/NUANCED tagging convention — the exact device Section 1 of this notebook spent time interrogating for Finding #4 and Finding #7. "CONFIRMED" implies a hypothesis was formally tested and held up. What I actually had was one observed pattern in a single 30k-row slice, no held-out validation of that specific claim, and — per the same logic that undercut the paper's position-based findings in Section 1 — no way to rule out that pages don't land in position tiers at random. Something driving both position and CTR together (topic quality, editorial priority, site authority) hasn't been ruled out.

Rewrite: In this dataset, weighted CTR falls monotonically from page_1 through deep position tiers, with no exceptions once the avg_position == 0 construction bug is corrected. This is consistent with the standard SEO expectation that click-through drops with rank — an observed, single-dataset pattern, not a confirmed causal test of the relationship. It inherits the same open question raised about the paper's own position-based findings: pages don't get assigned positions at random, so factors that drive both position and CTR together haven't been ruled out here either.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.